# 04 — یکسان‌سازی مکانی داده‌های CHARGED و UrbanEV

هدف این نوت‌بوک ساخت واحد مکانی قابل‌مقایسه برای آموزش و اعتبارسنجی خارجی است.

- CHARGED: دادهٔ تقاضا در سطح ایستگاه است.
- UrbanEV: دادهٔ تقاضا در سطح TAZ است.
- راهبرد: تجمیع ایستگاه‌های CHARGED در سلول‌های مکانی منظم، با مقیاسی مبتنی بر میانهٔ مساحت TAZهای UrbanEV.

دادهٔ UrbanEV فقط برای تعیین مقیاس مکانی استفاده می‌شود؛
در آموزش، تنظیم ابرپارامترها یا انتخاب مدل استفاده نخواهد شد.

In [1]:
# Prompt: Load processed CHARGED and UrbanEV data and define paths for spatial harmonization.

from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"

for directory in [
    INTERIM_DIR,
    PROCESSED_DIR,
    OUTPUT_TABLES_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

charged_modeling_base = pd.read_csv(
    PROCESSED_DIR / "charged_modeling_base.csv.gz",
    compression="gzip",
    parse_dates=["date"],
)

urbanev_taz_daily_targets = pd.read_csv(
    INTERIM_DIR / "urbanev_taz_day_targets.csv.gz",
    compression="gzip",
    parse_dates=["date"],
)

print("CHARGED shape:", charged_modeling_base.shape)
print("UrbanEV shape:", urbanev_taz_daily_targets.shape)

CHARGED shape: (518805, 36)
UrbanEV shape: (49775, 12)


## ۴-۱. تعیین مقیاس سلول مکانی از دادهٔ خارجی

مساحت TAZها فقط برای تعیین طول ضلع سلول استفاده می‌شود.
هدف‌های تقاضا و سایر متغیرهای UrbanEV در این مرحله استفاده نمی‌شوند.

In [2]:
# Prompt: Derive a transparent CHARGED grid-cell size from UrbanEV TAZ area distribution.

urbanev_taz_area = (
    urbanev_taz_daily_targets[
        ["taz_id", "area"]
    ]
    .drop_duplicates()
    .copy()
)

urbanev_taz_area["area"] = pd.to_numeric(
    urbanev_taz_area["area"],
    errors="coerce",
)

urbanev_taz_area = urbanev_taz_area.loc[
    urbanev_taz_area["area"] > 0
].copy()

median_taz_area_m2 = urbanev_taz_area["area"].median()

grid_cell_side_meters = int(
    round(np.sqrt(median_taz_area_m2) / 100) * 100
)

grid_scale_audit = pd.DataFrame(
    {
        "urbanEV_taz_count": [len(urbanev_taz_area)],
        "minimum_taz_area_m2": [
            urbanev_taz_area["area"].min()
        ],
        "median_taz_area_m2": [
            median_taz_area_m2
        ],
        "maximum_taz_area_m2": [
            urbanev_taz_area["area"].max()
        ],
        "derived_grid_cell_side_meters": [
            grid_cell_side_meters
        ],
        "derived_grid_cell_area_m2": [
            grid_cell_side_meters ** 2
        ],
    }
)

display(grid_scale_audit)

grid_scale_audit.to_csv(
    OUTPUT_TABLES_DIR
    / "cross_city_grid_scale_derivation.csv",
    index=False,
    encoding="utf-8-sig",
)

print(
    "Selected grid side length:",
    f"{grid_cell_side_meters:,} meters",
)

,urbanEV_taz_count,minimum_taz_area_m2,median_taz_area_m2,maximum_taz_area_m2,derived_grid_cell_side_meters,derived_grid_cell_area_m2
0,275,387839.4781,2317791.153,49933877.0,1500,2250000


Selected grid side length: 1,500 meters


## ۴-۲. تبدیل مختصات و تخصیص ایستگاه‌های CHARGED به سلول‌های ۱٫۵ کیلومتری

برای جلوگیری از خطای فاصله در مختصات طول و عرض جغرافیایی،
هر شهر در دستگاه مختصات UTM مناسب خود به متر تبدیل می‌شود.
فقط سلول‌هایی که دست‌کم یک ایستگاه شارژ دارند نگه داشته می‌شوند.

In [3]:
# Prompt: Assign each CHARGED station to an occupied 1.5-km projected grid cell.

from pyproj import CRS, Transformer

charged_site_locations = (
    charged_modeling_base[
        [
            "city_code",
            "city_name",
            "country_name",
            "site_id",
            "longitude",
            "latitude",
            "charger_num",
        ]
    ]
    .drop_duplicates()
    .copy()
)

def get_utm_epsg(longitude, latitude):
    utm_zone = int((longitude + 180) / 6) + 1

    if latitude >= 0:
        return 32600 + utm_zone

    return 32700 + utm_zone

charged_site_locations["utm_epsg"] = (
    charged_site_locations.apply(
        lambda row: get_utm_epsg(
            row["longitude"],
            row["latitude"],
        ),
        axis=1,
    )
)

charged_site_locations["x_meters"] = np.nan
charged_site_locations["y_meters"] = np.nan

for epsg_code in sorted(
    charged_site_locations["utm_epsg"].unique()
):
    city_mask = (
        charged_site_locations["utm_epsg"] == epsg_code
    )

    transformer = Transformer.from_crs(
        CRS.from_epsg(4326),
        CRS.from_epsg(epsg_code),
        always_xy=True,
    )

    projected_x, projected_y = transformer.transform(
        charged_site_locations.loc[
            city_mask,
            "longitude",
        ].to_numpy(),
        charged_site_locations.loc[
            city_mask,
            "latitude",
        ].to_numpy(),
    )

    charged_site_locations.loc[
        city_mask,
        "x_meters",
    ] = projected_x

    charged_site_locations.loc[
        city_mask,
        "y_meters",
    ] = projected_y

charged_site_locations["grid_x"] = (
    np.floor(
        charged_site_locations["x_meters"]
        / grid_cell_side_meters
    )
    .astype(int)
)

charged_site_locations["grid_y"] = (
    np.floor(
        charged_site_locations["y_meters"]
        / grid_cell_side_meters
    )
    .astype(int)
)

charged_site_locations["grid_id"] = (
    charged_site_locations["city_code"]
    + "_"
    + charged_site_locations["utm_epsg"].astype(str)
    + "_"
    + charged_site_locations["grid_x"].astype(str)
    + "_"
    + charged_site_locations["grid_y"].astype(str)
)

charged_grid_station_audit = (
    charged_site_locations
    .groupby(
        ["city_code", "city_name"],
        as_index=False,
    )
    .agg(
        station_count=("site_id", "nunique"),
        occupied_grid_cells=("grid_id", "nunique"),
        total_chargers=("charger_num", "sum"),
        mean_stations_per_cell=(
            "site_id",
            lambda x: round(
                x.nunique() / x.groupby(
                    charged_site_locations.loc[
                        x.index,
                        "grid_id",
                    ]
                ).ngroups,
                4,
            ),
        ),
    )
)

display(charged_grid_station_audit)

charged_site_locations.to_csv(
    INTERIM_DIR
    / "charged_site_grid_lookup.csv.gz",
    index=False,
    compression="gzip",
)

charged_grid_station_audit.to_csv(
    OUTPUT_TABLES_DIR
    / "charged_grid_station_assignment_audit.csv",
    index=False,
    encoding="utf-8-sig",
)

print(
    "Saved:",
    "data/interim/charged_site_grid_lookup.csv.gz",
)

,city_code,city_name,station_count,occupied_grid_cells,total_chargers,mean_stations_per_cell
0,AMS,Amsterdam,2449,87,3526,28.1494
1,JHB,Johannesburg,47,35,61,1.3429
2,LOA,Los Angeles,229,158,506,1.4494
3,MEL,Melbourne,63,58,64,1.0862
4,SPO,Sao Paulo,47,27,50,1.7407


Saved: data/interim/charged_site_grid_lookup.csv.gz


## ۴-۳. ساخت دیتاست روزانه–سلولی CHARGED

تقاضای ایستگاه‌ها در هر سلول و هر روز جمع می‌شود.
متغیر هدف اصلی، مجموع تقاضای روزانه تقسیم بر مجموع شارژرهای همان سلول است.

نام متغیر هدف «duration value» است؛ چون واحد خام duration در منبع
بدون تأیید مستندات، ساعت یا دقیقه فرض نمی‌شود.

In [4]:
# Prompt: Aggregate CHARGED station-day demand into comparable daily grid-cell observations.

charged_grid_static = (
    charged_site_locations
    .groupby(
        [
            "city_code",
            "city_name",
            "country_name",
            "grid_id",
        ],
        as_index=False,
    )
    .agg(
        grid_total_chargers=("charger_num", "sum"),
        grid_station_count=("site_id", "nunique"),
        grid_longitude=("longitude", "mean"),
        grid_latitude=("latitude", "mean"),
    )
)

charged_grid_features = (
    charged_modeling_base
    .merge(
        charged_site_locations[
            ["city_code", "site_id", "grid_id"]
        ],
        on=["city_code", "site_id"],
        how="left",
        validate="many_to_one",
    )
    .merge(
        charged_grid_static,
        on=[
            "city_code",
            "city_name",
            "country_name",
            "grid_id",
        ],
        how="left",
        validate="many_to_one",
    )
)

poi_feature_columns = sorted(
    [
        column
        for column in charged_grid_features.columns
        if column.startswith("poi_")
    ]
)

temporal_feature_columns = [
    "temp_mean",
    "humidity_mean",
    "windspeed_mean",
    "visibility_mean",
    "cloudcover_mean",
    "precip_total",
    "solarradiation_total",
    "has_precipitation",
    "recorded_month",
    "recorded_day_of_week",
    "recorded_day_of_year",
    "month_sin",
    "month_cos",
]

grid_daily_aggregation = {
    "daily_duration_hours": "sum",
    "grid_total_chargers": "first",
    "grid_station_count": "first",
    "grid_longitude": "first",
    "grid_latitude": "first",
}

for column in temporal_feature_columns + poi_feature_columns:
    grid_daily_aggregation[column] = "first"

charged_grid_day = (
    charged_grid_features
    .groupby(
        [
            "city_code",
            "city_name",
            "country_name",
            "grid_id",
            "date",
        ],
        as_index=False,
    )
    .agg(grid_daily_aggregation)
    .rename(
        columns={
            "daily_duration_hours": "daily_duration_value",
        }
    )
)

charged_grid_day[
    "daily_duration_per_charger"
] = (
    charged_grid_day["daily_duration_value"]
    / charged_grid_day["grid_total_chargers"]
)

charged_grid_day[
    "target_log_daily_duration_per_charger"
] = np.log1p(
    charged_grid_day["daily_duration_per_charger"]
)

charged_grid_day_audit = (
    charged_grid_day
    .groupby(
        ["city_code", "city_name"],
        as_index=False,
    )
    .agg(
        grid_day_records=("grid_id", "size"),
        unique_grid_cells=("grid_id", "nunique"),
        unique_dates=("date", "nunique"),
        mean_target=(
            "daily_duration_per_charger",
            "mean",
        ),
        median_target=(
            "daily_duration_per_charger",
            "median",
        ),
        zero_target_percent=(
            "daily_duration_per_charger",
            lambda x: 100 * (x == 0).mean(),
        ),
    )
)

display(charged_grid_day_audit)

charged_grid_day.to_csv(
    PROCESSED_DIR / "charged_grid_day_modeling_base.csv.gz",
    index=False,
    compression="gzip",
)

charged_grid_day_audit.to_csv(
    OUTPUT_TABLES_DIR
    / "charged_grid_day_construction_audit.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Shape:", charged_grid_day.shape)
print(
    "Saved:",
    "data/processed/charged_grid_day_modeling_base.csv.gz",
)

,city_code,city_name,grid_day_records,unique_grid_cells,unique_dates,mean_target,median_target,zero_target_percent
0,AMS,Amsterdam,15921,87,183,32.733788,17.016393,19.219898
1,JHB,Johannesburg,6405,35,183,4.396058,3.394737,12.365340
2,LOA,Los Angeles,28914,158,183,11.265905,8.916667,3.098845
3,MEL,Melbourne,10614,58,183,30.512806,21.833333,4.022989
4,SPO,Sao Paulo,4941,27,183,5.047999,3.455383,13.863590


Shape: (66795, 37)
Saved: data/processed/charged_grid_day_modeling_base.csv.gz


## ۴-۴. قرارداد ویژگی‌های قابل‌انتقال برای اعتبارسنجی خارجی

ویژگی‌های مکانی ریزدانهٔ POI در مدل اصلی اعتبارسنجی خارجی استفاده نمی‌شوند،
زیرا طبقه‌بندی و واحد مکانی آن‌ها بین دو منبع یکسان نیست.

ویژگی‌های مجاز:
- ظرفیت شارژ و تعداد ایستگاه
- دما و رطوبت
- وجود بارش
- روز هفته و چرخهٔ ماه

هدف در هر دو منبع: تقاضای روزانه به‌ازای هر شارژر.

In [5]:
# Prompt: Build the exact shared feature contract for CHARGED training and UrbanEV external validation.

urbanev_taz_day_temporal_features = pd.read_csv(
    INTERIM_DIR
    / "urbanev_taz_day_temporal_features.csv.gz",
    compression="gzip",
    parse_dates=["date"],
)

portable_feature_columns = [
    "log_total_chargers",
    "log_station_count",
    "temp_mean",
    "humidity_mean",
    "has_precipitation",
    "recorded_day_of_week",
    "month_sin",
    "month_cos",
]

charged_portable_modeling = (
    charged_grid_day
    .copy()
)

charged_portable_modeling["log_total_chargers"] = np.log1p(
    charged_portable_modeling["grid_total_chargers"]
)

charged_portable_modeling["log_station_count"] = np.log1p(
    charged_portable_modeling["grid_station_count"]
)

urbanev_portable_modeling = (
    urbanev_taz_day_temporal_features
    .copy()
)

urbanev_portable_modeling["log_total_chargers"] = np.log1p(
    urbanev_portable_modeling["total_chargers"]
)

urbanev_portable_modeling["log_station_count"] = np.log1p(
    urbanev_portable_modeling["station_count"]
)

urbanev_portable_modeling[
    "target_log_daily_duration_per_charger"
] = np.log1p(
    urbanev_portable_modeling[
        "daily_duration_per_charger"
    ]
)

feature_contract_audit = pd.DataFrame(
    {
        "feature": portable_feature_columns,
        "charged_missing_values": [
            int(charged_portable_modeling[feature].isna().sum())
            for feature in portable_feature_columns
        ],
        "urbanev_missing_values": [
            int(urbanev_portable_modeling[feature].isna().sum())
            for feature in portable_feature_columns
        ],
        "charged_minimum": [
            charged_portable_modeling[feature].min()
            for feature in portable_feature_columns
        ],
        "charged_maximum": [
            charged_portable_modeling[feature].max()
            for feature in portable_feature_columns
        ],
        "urbanev_minimum": [
            urbanev_portable_modeling[feature].min()
            for feature in portable_feature_columns
        ],
        "urbanev_maximum": [
            urbanev_portable_modeling[feature].max()
            for feature in portable_feature_columns
        ],
    }
)

display(feature_contract_audit)

charged_portable_modeling.to_csv(
    PROCESSED_DIR
    / "charged_grid_day_portable_modeling.csv.gz",
    index=False,
    compression="gzip",
)

urbanev_portable_modeling.to_csv(
    PROCESSED_DIR
    / "urbanev_taz_day_portable_modeling.csv.gz",
    index=False,
    compression="gzip",
)

feature_contract_audit.to_csv(
    OUTPUT_TABLES_DIR
    / "cross_city_portable_feature_contract.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Shared portable features:", portable_feature_columns)
print(
    "Saved portable modeling files in data/processed/",
)

,feature,charged_missing_values,urbanev_missing_values,charged_minimum,charged_maximum,urbanev_minimum,urbanev_maximum
0,log_total_chargers,0,0,0.693147,5.236442e+00,1.609438e+00,5.924256
1,log_station_count,0,0,0.693147,4.820282e+00,6.931472e-01,3.737670
2,temp_mean,0,0,3.237500,2.756667e+01,1.038750e+01,31.137500
3,humidity_mean,0,0,13.880417,9.744167e+01,2.908333e+01,94.416667
4,has_precipitation,0,0,0.000000,1.000000e+00,0.000000e+00,1.000000
5,recorded_day_of_week,0,0,0.000000,6.000000e+00,0.000000e+00,6.000000
6,month_sin,0,0,-1.000000,8.660254e-01,-1.000000e+00,0.866025
7,month_cos,0,0,-1.000000,-1.836970e-16,-1.836970e-16,1.000000


Shared portable features: ['log_total_chargers', 'log_station_count', 'temp_mean', 'humidity_mean', 'has_precipitation', 'recorded_day_of_week', 'month_sin', 'month_cos']
Saved portable modeling files in data/processed/


## ۴-۵. سنجش انتقال‌پذیری دامنهٔ ویژگی‌ها

برای هر ویژگی، درصد مشاهده‌های UrbanEV که بیرون از کمینه تا بیشینهٔ
مشاهده‌شده در CHARGED قرار دارند محاسبه می‌شود.

این مرحله صرفاً ممیزی انتقال‌پذیری است و برای حذف هیچ مشاهده‌ای استفاده نمی‌شود.

In [6]:
# Prompt: Quantify UrbanEV observations that require extrapolation beyond CHARGED feature ranges.

range_shift_rows = []

for feature in portable_feature_columns:
    charged_minimum = charged_portable_modeling[
        feature
    ].min()

    charged_maximum = charged_portable_modeling[
        feature
    ].max()

    urbanev_values = urbanev_portable_modeling[
        feature
    ]

    below_training_range = (
        urbanev_values < charged_minimum
    )

    above_training_range = (
        urbanev_values > charged_maximum
    )

    outside_training_range = (
        below_training_range
        | above_training_range
    )

    range_shift_rows.append(
        {
            "feature": feature,
            "charged_minimum": charged_minimum,
            "charged_maximum": charged_maximum,
            "urbanev_below_range_count": int(
                below_training_range.sum()
            ),
            "urbanev_above_range_count": int(
                above_training_range.sum()
            ),
            "urbanev_outside_range_count": int(
                outside_training_range.sum()
            ),
            "urbanev_outside_range_percent": round(
                100 * outside_training_range.mean(),
                4,
            ),
        }
    )

urbanev_range_shift_audit = pd.DataFrame(
    range_shift_rows
)

outside_any_feature = pd.Series(
    False,
    index=urbanev_portable_modeling.index,
)

for feature in portable_feature_columns:
    charged_minimum = charged_portable_modeling[
        feature
    ].min()

    charged_maximum = charged_portable_modeling[
        feature
    ].max()

    outside_any_feature = (
        outside_any_feature
        | (
            urbanev_portable_modeling[feature]
            < charged_minimum
        )
        | (
            urbanev_portable_modeling[feature]
            > charged_maximum
        )
    )

overall_range_shift_audit = pd.DataFrame(
    {
        "urbanev_records": [
            len(urbanev_portable_modeling)
        ],
        "records_outside_at_least_one_range": [
            int(outside_any_feature.sum())
        ],
        "records_outside_at_least_one_range_percent": [
            round(
                100 * outside_any_feature.mean(),
                4,
            )
        ],
    }
)

display(urbanev_range_shift_audit)
display(overall_range_shift_audit)

urbanev_range_shift_audit.to_csv(
    OUTPUT_TABLES_DIR
    / "cross_city_urbanev_feature_range_shift.csv",
    index=False,
    encoding="utf-8-sig",
)

overall_range_shift_audit.to_csv(
    OUTPUT_TABLES_DIR
    / "cross_city_urbanev_overall_range_shift.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Saved feature-range transferability audits in outputs/tables/")

,feature,charged_minimum,charged_maximum,urbanev_below_range_count,urbanev_above_range_count,urbanev_outside_range_count,urbanev_outside_range_percent
0,log_total_chargers,0.693147,5.236442e+00,0,1991,1991,4.0000
1,log_station_count,0.693147,4.820282e+00,0,0,0,0.0000
2,temp_mean,3.237500,2.756667e+01,0,9350,9350,18.7845
3,humidity_mean,13.880417,9.744167e+01,0,0,0,0.0000
4,has_precipitation,0.000000,1.000000e+00,0,0,0,0.0000
5,recorded_day_of_week,0.000000,6.000000e+00,0,0,0,0.0000
6,month_sin,-1.000000,8.660254e-01,0,0,0,0.0000
7,month_cos,-1.000000,-1.836970e-16,0,41525,41525,83.4254


,urbanev_records,records_outside_at_least_one_range,records_outside_at_least_one_range_percent
0,49775,49511,99.4696


Saved feature-range transferability audits in outputs/tables/


## ۴-۶. بازنگری قرارداد ویژگی‌ها پس از ممیزی انتقال‌پذیری

چرخهٔ ماه از مدل اصلی اعتبارسنجی خارجی حذف می‌شود؛
زیرا بازه‌های زمانی دو منبع هم‌پوشانی فصلی کافی ندارند.

مدل اصلی خارجی بر متغیرهای مشترک و قابل‌انتقال ظرفیت، آب‌وهوا و روز هفته تکیه دارد.

In [7]:
# Prompt: Define the final extrapolation-aware feature set for external validation.

external_validation_features = [
    "log_total_chargers",
    "log_station_count",
    "temp_mean",
    "humidity_mean",
    "has_precipitation",
    "recorded_day_of_week",
]

internal_extended_features = (
    external_validation_features
    + [
        "month_sin",
        "month_cos",
    ]
)

external_feature_decision = pd.DataFrame(
    [
        {
            "feature": feature,
            "used_for_external_validation": (
                feature in external_validation_features
            ),
            "used_for_internal_extended_model": (
                feature in internal_extended_features
            ),
        }
        for feature in portable_feature_columns
    ]
)

revised_range_shift_rows = []

for feature in external_validation_features:
    charged_minimum = charged_portable_modeling[
        feature
    ].min()

    charged_maximum = charged_portable_modeling[
        feature
    ].max()

    urban_values = urbanev_portable_modeling[feature]

    outside_range = (
        (urban_values < charged_minimum)
        | (urban_values > charged_maximum)
    )

    revised_range_shift_rows.append(
        {
            "feature": feature,
            "urbanev_outside_range_count": int(
                outside_range.sum()
            ),
            "urbanev_outside_range_percent": round(
                100 * outside_range.mean(),
                4,
            ),
        }
    )

revised_range_shift_audit = pd.DataFrame(
    revised_range_shift_rows
)

display(external_feature_decision)
display(revised_range_shift_audit)

external_feature_decision.to_csv(
    OUTPUT_TABLES_DIR
    / "cross_city_final_feature_decision.csv",
    index=False,
    encoding="utf-8-sig",
)

revised_range_shift_audit.to_csv(
    OUTPUT_TABLES_DIR
    / "cross_city_final_external_range_shift.csv",
    index=False,
    encoding="utf-8-sig",
)

print(
    "Final external features:",
    external_validation_features,
)

,feature,used_for_external_validation,used_for_internal_extended_model
0,log_total_chargers,True,True
1,log_station_count,True,True
2,temp_mean,True,True
3,humidity_mean,True,True
4,has_precipitation,True,True
5,recorded_day_of_week,True,True
6,month_sin,False,True
7,month_cos,False,True


,feature,urbanev_outside_range_count,urbanev_outside_range_percent
0,log_total_chargers,1991,4.0000
1,log_station_count,0,0.0000
2,temp_mean,9350,18.7845
3,humidity_mean,0,0.0000
4,has_precipitation,0,0.0000
5,recorded_day_of_week,0,0.0000


Final external features: ['log_total_chargers', 'log_station_count', 'temp_mean', 'humidity_mean', 'has_precipitation', 'recorded_day_of_week']
